<a href="https://colab.research.google.com/github/luizatfilip10/Analyzing_data_A3/blob/main/Assignment3_AD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analyzing data Assignment 3
Luiza Filip
s5183685

## 1. Installing required packages

In [1]:
!pip install -q ollama pandas scikit-learn tqdm

## 2. Import libraries

 ##### ollama: used to prompt the LLM

 ##### pandas: used for loading and manipulating CSV data
 ##### tqdm: shows a progress bar during slow inference
 ##### sklearn: used for evaluation metrics


In [2]:
import ollama
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

In [3]:
!sudo apt update && sudo apt install pciutils lshw

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,482 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,972 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,310 kB]
Hit:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64

In [4]:
!apt-get install -y zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 54 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (1,041 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122387 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


## 3. Installing Ollama

In [29]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [30]:
!nohup ollama serve > ollama.log 2>&1 &

## 4. Download the gemma3 model

In [31]:
!ollama pull gemma3

### List downloaded models

In [32]:
!ollama list

NAME             ID              SIZE      MODIFIED       
gemma3:latest    a2af6cc3eb7f    3.3 GB    18 seconds ago    


## 5. Define file paths

In [33]:
train_path = "/content/genreLyrics_train.csv"
test_path = "/content/genreLyrics_test.csv"

## 6. Load the Train and Test data

- These files are tab-separated, so we use sep="\t".
- The train set will be used for few-shot examples.
- The test set will be used for final evaluation.

In [34]:
df_train = pd.read_csv(train_path, sep="\t")
df_test = pd.read_csv(test_path, sep="\t")

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)

Train shape: (5000, 3)
Test shape: (1500, 3)


## 7. Inspect the data
- Check the first rows and genre distribution.
- This helps confirm that the files loaded correctly.

In [35]:
print("Train columns:", df_train.columns.tolist())
print("\nFirst 5 rows of training data:")
print(df_train.head())

print("\nGenre distribution in training data:")
print(df_train["genre"].value_counts())

Train columns: ['Unnamed: 0', 'genre', 'lyrics']

First 5 rows of training data:
   Unnamed: 0       genre                                             lyrics
0      263935        Rock  I knew a man, called him Sandy Cane\nFew folks...
1       64235     Country  (Gary Harrison - Kent Robbins - Gene Miller)\n...
2      197695        Rock  You think its right when you see my reaction\n...
3      274428  Electronic  Darkness, you are gentle when you kiss me\nYou...
4      116870        Rock  Cold fire, you've got everything but cold fire...

Genre distribution in training data:
genre
Rock          2213
Pop            843
Hip-Hop        518
Metal          498
Country        302
Jazz           166
Electronic     156
Other          111
Indie           74
R&B             67
Folk            52
Name: count, dtype: int64


## 8. Basic data cleaning
- Remove missing values in the key columns and ensure lyrics are strings.


In [36]:
df_train = df_train.dropna(subset=["genre", "lyrics"]).copy()
df_test = df_test.dropna(subset=["genre", "lyrics"]).copy()

df_train["lyrics"] = df_train["lyrics"].astype(str).str.strip()
df_test["lyrics"] = df_test["lyrics"].astype(str).str.strip()

# Remove rows with empty lyrics
df_train = df_train[df_train["lyrics"] != ""].copy()
df_test = df_test[df_test["lyrics"] != ""].copy()

print("Train shape after cleaning:", df_train.shape)
print("Test shape after cleaning:", df_test.shape)

Train shape after cleaning: (5000, 3)
Test shape after cleaning: (1500, 3)


## 9. Define label of genres
- We extract the official labels from the dataset itself.
- This is safer than typing them manually.

In [37]:
genres = sorted(df_train["genre"].unique().tolist())

print("Genres:")
print(genres)

Genres:
['Country', 'Electronic', 'Folk', 'Hip-Hop', 'Indie', 'Jazz', 'Metal', 'Other', 'Pop', 'R&B', 'Rock']


## 10. Define helper functions

We define:
- the model name
- a function to query Ollama
- a function to clean/normalize predictions

The cleaning step is important because LLMs often return free-form text instead of exact labels.

In [38]:
MODEL_NAME = "gemma3"

In [39]:

def get_prediction(prompt, model=MODEL_NAME):
    """
    Send a prompt to the Ollama model and return the raw text response.

    Parameters:
        prompt (str): The prompt sent to the model.
        model (str): Name of the Ollama model.

    Returns:
        str: Raw model output.
    """
    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    return response["message"]["content"]

In [40]:
def clean_prediction(pred):
    """
    Convert raw model output into one of the official genre labels.
    """
    pred = str(pred).strip().lower()

    replacements = {
        "hip hop": "Hip-Hop",
        "hip-hop": "Hip-Hop",
        "hiphop": "Hip-Hop",
        "rap": "Hip-Hop",
        "r&b": "R&B",
        "rnb": "R&B",
        "rhythm and blues": "R&B",
        "electronic": "Electronic",
        "edm": "Electronic"
    }

    for key, value in replacements.items():
        if key in pred:
            return value

    for g in genres:
        if g.lower() in pred:
            return g

    return "Other"

## 11. Build few-shot examples from the training set

Important: the model is **not trained** on this data.  
The training split is only used to create labeled examples inside the prompt.

In [41]:
# Select one random training example per genre
few_shot_examples = (
    df_train.groupby("genre", group_keys=False)
    .apply(lambda x: x.sample(1, random_state=42))
    .reset_index(drop=True)
)

print("Few-shot examples selected:")
print(few_shot_examples[["genre"]])

Few-shot examples selected:
         genre
0      Country
1   Electronic
2         Folk
3      Hip-Hop
4        Indie
5         Jazz
6        Metal
7        Other
8          Pop
9          R&B
10        Rock


/tmp/ipykernel_11605/3155735071.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(1, random_state=42))


In [42]:
# Build few-shot context
few_shot_context = "Classify these song lyrics into one of: " + ", ".join(genres) + "\n\nExamples:\n\n"

for _, row in few_shot_examples.iterrows():
    few_shot_context += (
        f"Lyrics: {row['lyrics'][:120]}\n"
        f"Genre: {row['genre']}\n"
        f"---\n"
    )

print(few_shot_context[:1000])

Classify these song lyrics into one of: Country, Electronic, Folk, Hip-Hop, Indie, Jazz, Metal, Other, Pop, R&B, Rock

Examples:

Lyrics: Take me back to Tulsa, I'm too young to marry
Take me back to Tulsa, I'm too young to marry
You see that girl with the r
Genre: Country
---
Lyrics: I rap, I
I can see electricity
I rap, I
I can see electricity
I rap, I
I can see electricity
...
Genre: Electronic
---
Lyrics: Niin kauan kun ma muistan
on ollut humppaa
sen villit rytmit raikui
jo nuijasodassa
sit oli paljon
ja se oli mahtavaa
m 
Genre: Folk
---
Lyrics: Pam, the funkstress, you're so tight
Just scratch me up and make it right
I'm caught up in your flow tonight
So you can 
Genre: Hip-Hop
---
Lyrics: I love my baby, my baby loves me
Don't know nobody as happy as we
She's only twenty and I'm twenty one
We never worry, w
Genre: Indie
---
Lyrics: Instrumental
Genre: Jazz
---
Lyrics: The days when ancient blood
Will awake in the hearts of white men and women
Our banners will rise to the sky
An

## 12. Define zero-shot and few-shot prompts


In [43]:
def zero_shot_prompt(lyrics):
    return f"""
Classify these song lyrics into one of: {', '.join(genres)}.
Provide ONLY the genre name as output.

Lyrics: {lyrics[:120]}
Genre:
""".strip()


def few_shot_prompt(lyrics):
    return f"""
{few_shot_context}
Now classify these new lyrics into one of: {', '.join(genres)}.
Provide ONLY the genre name as output.

Lyrics: {lyrics[:120]}
Genre:
""".strip()

## 13. Evaluation set



In [44]:
eval_df = df_test.copy().reset_index(drop=True)

print("Evaluation size:", len(eval_df))

Evaluation size: 1500


## 14. Run zero-shot and few-shot inference on the evaluation set

In [45]:
zero_raws = []
zero_preds = []
few_raws = []
few_preds = []

for i, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    # Zero-shot prediction
    z_raw = get_prediction(zero_shot_prompt(row["lyrics"]))
    z_clean = clean_prediction(z_raw)

    # Few-shot prediction
    f_raw = get_prediction(few_shot_prompt(row["lyrics"]))
    f_clean = clean_prediction(f_raw)

    # Save outputs in memory
    zero_raws.append(z_raw)
    zero_preds.append(z_clean)
    few_raws.append(f_raw)
    few_preds.append(f_clean)

    # Save checkpoint every 50 songs
    if (i + 1) % 50 == 0:
        checkpoint_df = eval_df.iloc[:i+1].copy()
        checkpoint_df["zero_raw"] = zero_raws
        checkpoint_df["zero_pred"] = zero_preds
        checkpoint_df["few_raw"] = few_raws
        checkpoint_df["few_pred"] = few_preds
        checkpoint_df.to_csv("checkpoint_predictions.csv", index=False)
        print(f"Checkpoint saved after {i+1} songs")

# Save final predictions to dataframe
eval_df["zero_raw"] = zero_raws
eval_df["zero_pred"] = zero_preds
eval_df["few_raw"] = few_raws
eval_df["few_pred"] = few_preds

print("Inference completed.")

  3%|▎         | 50/1500 [02:52<28:50,  1.19s/it]

Checkpoint saved after 50 songs


  7%|▋         | 100/1500 [03:59<28:56,  1.24s/it]

Checkpoint saved after 100 songs


 10%|█         | 150/1500 [05:05<27:59,  1.24s/it]

...

Checkpoint saved after 1400 songs


 97%|█████████▋| 1450/1500 [34:12<01:14,  1.49s/it]

Checkpoint saved after 1450 songs


100%|██████████| 1500/1500 [35:19<00:00,  1.41s/it]

Checkpoint saved after 1500 songs
Inference completed.


## 15. Inspect model outputs before evaluation

In [46]:
# View a few examples of raw and cleaned predictions
print(eval_df[["genre", "zero_raw", "zero_pred", "few_raw", "few_pred"]].head(10))

        genre   zero_raw zero_pred       few_raw    few_pred
0     Hip-Hop  Hip-Hop\n   Hip-Hop     Hip-Hop\n     Hip-Hop
1     Country        Pop       Pop     Country\n     Country
2        Rock      Pop\n       Pop  Electronic\n  Electronic
3  Electronic       Folk      Folk        Folk\n        Folk
4         Pop      Pop\n       Pop        Rock\n        Rock
5       Indie     Folk\n      Folk     Country\n     Country
6        Rock    Indie\n     Indie       Indie\n       Indie
7        Rock    Indie\n     Indie        Rock\n        Rock
8        Folk     Folk\n      Folk     Country\n     Country
9        Rock      Pop\n       Pop       Other\n       Other


In [47]:
# Check the most frequent raw outputs
print("Most common ZERO-SHOT raw outputs:")
print(eval_df["zero_raw"].value_counts().head(20))

print("\nMost common FEW-SHOT raw outputs:")
print(eval_df["few_raw"].value_counts().head(20))

Most common ZERO-SHOT raw outputs:
zero_raw
Pop\n           277
Folk\n          268
Country\n       208
Rock\n          176
Hip-Hop\n       129
Folk            104
Indie\n          98
Other\n          71
Pop              45
Metal\n          33
Hip-Hop          11
Indie            10
Latin Pop\n      10
Electronic\n      8
Rock \n           8
Indie \n          6
Other             6
Jazz\n            5
Rock              5
R&B\n             4
Name: count, dtype: int64

Most common FEW-SHOT raw outputs:
few_raw
Country\n       329
Pop\n           281
Folk\n          194
Other\n         188
Rock\n          153
Hip-Hop\n       116
Indie\n          61
Metal\n          44
Folk             42
Electronic\n     27
Hip-Hop          24
Other            15
Pop              11
Latin Pop\n       5
R&B               2
Blues\n           1
Funk\n            1
Indie             1
Reggae\n          1
Jazz\n            1
Name: count, dtype: int64


## 16. Final performance reports

In [48]:
for mode, col in [("ZERO-SHOT", "zero_pred"), ("FEW-SHOT", "few_pred")]:
    print(f"\n{'='*20} {mode} PERFORMANCE {'='*20}")
    print("Macro Precision:", precision_score(eval_df["genre"], eval_df[col], average="macro", zero_division=0))
    print("Macro Recall:   ", recall_score(eval_df["genre"], eval_df[col], average="macro", zero_division=0))
    print("Macro F1:       ", f1_score(eval_df["genre"], eval_df[col], average="macro", zero_division=0))
    print("\nDetailed classification report:")
    print(classification_report(eval_df["genre"], eval_df[col], zero_division=0))


==================== ZERO-SHOT PERFORMANCE ====================
Macro Precision: 0.25057585821830064
Macro Recall:    0.20385341377281857
Macro F1:        0.16588446346066524

Detailed classification report:
              precision    recall  f1-score   support

     Country       0.18      0.42      0.25        89
  Electronic       0.22      0.04      0.06        57
        Folk       0.02      0.41      0.04        17
     Hip-Hop       0.61      0.52      0.56       163
       Indie       0.01      0.06      0.02        16
        Jazz       0.17      0.02      0.04        43
       Metal       0.70      0.15      0.25       151
       Other       0.04      0.10      0.05        31
         Pop       0.28      0.36      0.31       255
         R&B       0.00      0.00      0.00        23
        Rock       0.55      0.16      0.25       655

    accuracy                           0.24      1500
   macro avg       0.25      0.20      0.17      1500
weighted avg       0.45      0.24

## 17. Create a summary table

In [49]:
results_summary = pd.DataFrame({
    "strategy": ["Zero-shot", "Few-shot"],
    "precision_macro": [
        precision_score(eval_df["genre"], eval_df["zero_pred"], average="macro", zero_division=0),
        precision_score(eval_df["genre"], eval_df["few_pred"], average="macro", zero_division=0)
    ],
    "recall_macro": [
        recall_score(eval_df["genre"], eval_df["zero_pred"], average="macro", zero_division=0),
        recall_score(eval_df["genre"], eval_df["few_pred"], average="macro", zero_division=0)
    ],
    "f1_macro": [
        f1_score(eval_df["genre"], eval_df["zero_pred"], average="macro", zero_division=0),
        f1_score(eval_df["genre"], eval_df["few_pred"], average="macro", zero_division=0)
    ]
})

print(results_summary)

    strategy  precision_macro  recall_macro  f1_macro
0  Zero-shot         0.250576      0.203853  0.165884
1   Few-shot         0.221626      0.204166  0.160892


## 18. Save results

In [50]:
# Save outputs so they can be reused later
eval_df.to_csv("genre_predictions_results.csv", index=False)
results_summary.to_csv("results_summary.csv", index=False)

print("All result files saved successfully.")

All result files saved successfully.
